# 07-4. 세션·쿠키·인증 예제

## Goal

- 쿠키 속성을 값과 분리해 확인합니다.
- 인증 헤더를 로그에서 마스킹합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

실제 계정·토큰 없이 합성 헤더만 사용합니다.


## Steps

### 쿠키 정책과 헤더 마스킹

민감한 값은 출력하지 않고 보안 속성과 헤더 존재 여부만 기록합니다.


In [1]:
from http.cookies import SimpleCookie


def cookie_policy(set_cookie: str) -> dict:
    jar = SimpleCookie()
    jar.load(set_cookie)
    if len(jar) != 1:
        raise ValueError("쿠키 하나만 기대합니다")
    name, morsel = next(iter(jar.items()))
    return {
        "name": name,
        "httponly": bool(morsel["httponly"]),
        "secure": bool(morsel["secure"]),
        "samesite": morsel["samesite"].lower(),
    }


def redact_headers(headers: dict[str, str]) -> dict[str, str]:
    sensitive = {"authorization", "cookie", "set-cookie"}
    return {name: "[가림]" if name.lower() in sensitive else value for name, value in headers.items()}


policy = cookie_policy("session=training-value; HttpOnly; SameSite=Lax")
safe_headers = redact_headers({"Authorization": "Bearer training-token", "Accept": "application/json"})
print(policy)
print(safe_headers)


{'name': 'session', 'httponly': True, 'secure': False, 'samesite': 'lax'}
{'Authorization': '[가림]', 'Accept': 'application/json'}


## Checks

쿠키 값과 토큰이 출력 결과에서 제거되었는지 확인합니다.


In [2]:
assert policy == {"name": "session", "httponly": True, "secure": False, "samesite": "lax"}
assert safe_headers["Authorization"] == "[가림]"
assert "training-token" not in repr(safe_headers)
print("민감정보 비기록 검사 통과")


민감정보 비기록 검사 통과


## Next Steps

운영 환경의 세션 쿠키에는 HTTPS를 전제로 `Secure` 속성도 적용합니다.
